# GPT-2 推論エンジン ウォークスルー（PyTorch 版）

解説記事: [【簡約版】GPT-2 推論エンジン入門](https://zenn.dev/7shi/articles/20260330-gpt2-inference)（本編: [GPT-2 推論エンジン入門](https://zenn.dev/7shi/books/20260319-gpt2)）

記事の各セクションに対応するコードを、このリポジトリの PyTorch 版実装 (`my_gpt2_torch/`) を使って動かします。解説は記事を参照してください。記事のコードは NumPy 版ですが、PyTorch 版も同じロジック・構造です（`np.` → `torch.` などの置き換えのみ）。

**前提**: プロジェクトルートで次を実行済みであること（ノートブックの作業ディレクトリもプロジェクトルート）。

```bash
uv sync
make download
```

PyTorch は extras（`torch`）としてインストールします。

```bash
pip install ".[torch]" jupyterlab && jupyter lab walkthrough.ipynb
# uv の場合
uv run --group notebook --extra torch jupyter lab walkthrough.ipynb
```

# セットアップ

In [2]:
import torch

from my_gpt2.tokenizer import Tokenizer
from my_gpt2.spiece import SentencePieceTokenizer
from my_gpt2_torch.loader import load_gpt2_weights
from my_gpt2_torch.model import softmax, gelu

model_id = "openai-community/gpt2"
tokenizer = Tokenizer(model_id)
model = load_gpt2_weights(model_id)
wte, wpe, ln_f, blocks = model.wte, model.wpe, model.ln_f, model.blocks

# テキスト生成

記事のコマンド（`uv run my-gpt2-torch ...`）と同じ処理を、`generate` 関数で呼び出します。

In [3]:
from my_gpt2_torch.generate import generate

# uv run my-gpt2-torch "Once upon a time"
_ = generate("Once upon a time", model=model, tokenizer=tokenizer)

Once upon a time, that all code was first committed to GC v3. Here is the record of successfully collected classes:

GCEv3_News serial


In [4]:
# uv run my-gpt2-torch -n 20 -m rinna/japanese-gpt2-small "吾輩は猫で"
rinna_id = "rinna/japanese-gpt2-small"
_ = generate("吾輩は猫で", n_tokens_to_generate=20, model_id=rinna_id,
             model=load_gpt2_weights(rinna_id), tokenizer=SentencePieceTokenizer(rinna_id))

吾輩は猫で出来ているので、君を一人でも安心させなければならない 坂本龍馬、ミヤマタ


生成は確率的なため、実行するたびに結果が変わります。`torch.manual_seed` で乱数シードを固定し、`temperature` `top_k` `top_p` で振る舞いを調整できます（CLI の `-s` `-t` `-k` `-p` に相当）。

In [5]:
torch.manual_seed(42)
_ = generate("Once upon a time", temperature=0.7, top_k=40, model=model, tokenizer=tokenizer)

Once upon a time, there was a certain person in the world who believed that the whole world was out to get him. The moment they found him, they were all


# 推論パイプラインの全体像

## モデルの重み

In [6]:
from safetensors.torch import load_file

weights = load_file(f"weights/{model_id}/model.safetensors")
for key in sorted(weights):
    if key.startswith("h.") and not key.startswith("h.0."):
        continue  # h.1 〜 h.11 は h.0 と同じ構造
    print(f"{key:28s} {str(tuple(weights[key].shape)):14s}")
print("ブロック数:", len({k.split(".")[1] for k in weights if k.startswith("h.")}))
del weights

h.0.attn.bias                (1, 1, 1024, 1024)
h.0.attn.c_attn.bias         (2304,)       
h.0.attn.c_attn.weight       (768, 2304)   
h.0.attn.c_proj.bias         (768,)        
h.0.attn.c_proj.weight       (768, 768)    
h.0.ln_1.bias                (768,)        
h.0.ln_1.weight              (768,)        
h.0.ln_2.bias                (768,)        
h.0.ln_2.weight              (768,)        
h.0.mlp.c_fc.bias            (3072,)       
h.0.mlp.c_fc.weight          (768, 3072)   
h.0.mlp.c_proj.bias          (768,)        
h.0.mlp.c_proj.weight        (3072, 768)   
ln_f.bias                    (768,)        
ln_f.weight                  (768,)        
wpe.weight                   (1024, 768)   
wte.weight                   (50257, 768)  
ブロック数: 12


## コア推論コード

In [7]:
text = "The capital of France is"

# Step 0: トークナイザー — テキストをトークンIDの列に変換
input_ids = tokenizer.encode(text)

# Step 1: Embedding — トークンIDをベクトルに変換し、位置情報を加算
x = wte[input_ids] + wpe[torch.arange(len(input_ids))]

# Step 2: Transformer Block × 12 — 文脈理解と特徴変換を繰り返す
for block in blocks:
    y = block.ln_1(x)  # 正規化
    y = block.attn(y)  # 文脈理解
    x = x + y          # 残差接続（加算）
    z = block.ln_2(x)  # 正規化
    z = block.mlp(z)   # 特徴変換
    x = x + z          # 残差接続（加算）

# Step 3: 最終 LayerNorm — 出力前の正規化
x = ln_f(x)

# Step 4: LM Head — 全語彙の埋め込みベクトルとの内積（ロジット）を一括計算
logits = x @ wte.T

# Step 5: サンプリング — 確率分布から次のトークンを選択
probs = softmax(logits[-1])
next_id = torch.argmax(probs).item()

print(f"入力: {text!r} -> {input_ids}")
print(f"次のトークン: {tokenizer.decode([next_id])!r} (確率 {probs[next_id].item():.3f})")

入力: 'The capital of France is' -> [464, 3139, 286, 4881, 318]
次のトークン: ' the' (確率 0.085)


In [8]:
# モデル本体の呼び出しと一致することを確認
print(torch.allclose(logits, model(torch.tensor(input_ids))))

True


# トークナイザー

## BPE（Byte Pair Encoding）

In [9]:
import regex

# 1. 事前分割
words = regex.findall(tokenizer.pat, "Hello, world!")
print("事前分割:", words)

# 2. バイトマッピング（空白 -> Ġ など）
mapped = ["".join(tokenizer.byte_encoder[b] for b in w.encode("utf-8")) for w in words]
print("バイトマッピング:", mapped)

事前分割: ['Hello', ',', ' world', '!']
バイトマッピング: ['Hello', ',', 'Ġworld', '!']


In [10]:
# 3. BPE マージ — "Hello" のマージ過程
from my_gpt2.tokenizer import get_pairs

word = tuple("Hello")
print("初期:", list(word))
step = 0
while len(word) > 1:
    pair = min(get_pairs(word), key=lambda p: tokenizer.bpe_ranks.get(p, float("inf")))
    if pair not in tokenizer.bpe_ranks:
        break
    step += 1
    merged, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
            merged.append(pair[0] + pair[1]); i += 2
        else:
            merged.append(word[i]); i += 1
    word = tuple(merged)
    print(f"step{step}: {list(pair)} (rank={tokenizer.bpe_ranks[pair]}) -> {list(word)}")

# 4. ID マッピング
print("ID:", tokenizer.encoder["Hello"], "/ encode:", tokenizer.encode("Hello"))

初期: ['H', 'e', 'l', 'l', 'o']
step1: ['l', 'l'] (rank=41) -> ['H', 'e', 'll', 'o']
step2: ['e', 'll'] (rank=439) -> ['H', 'ell', 'o']
step3: ['ell', 'o'] (rank=10853) -> ['H', 'ello']
step4: ['H', 'ello'] (rank=15240) -> ['Hello']
ID: 15496 / encode: [15496]


In [11]:
# 日本語はバイト単位に分解されるため非効率
s = "日本語"
ids = tokenizer.encode(s)
print(len(s), "文字 ->", len(ids), "トークン:", ids)

3 文字 -> 6 トークン: [33768, 98, 17312, 105, 45739, 252]


## SentencePiece（Unigram モデル）

In [12]:
sp = SentencePieceTokenizer("rinna/japanese-gpt2-small")
text_ja = "日本語"
print("正規化:", repr(sp._normalize(text_ja)))
for p in ["▁", "日本語"]:
    print(f"score({p!r}) = {sp._piece_to_score[p]:.4f}")
print("合計:", sp._piece_to_score["▁"] + sp._piece_to_score["日本語"])

正規化: '▁日本語'
score('▁') = -3.5238
score('日本語') = -9.9070
合計: -13.430808544158936


In [13]:
import math

# Viterbi アルゴリズム（my_gpt2/spiece.py の encode と同じ処理）
normalized = sp._normalize(text_ja)
n = len(normalized)
best = [(-math.inf, -1, None)] * (n + 1)
best[0] = (0.0, -1, None)
for i in range(n):
    for j in range(i + 1, n + 1):
        piece = normalized[i:j]
        if piece in sp._piece_to_score and best[i][0] > -math.inf:
            score = best[i][0] + sp._piece_to_score[piece]
            if score > best[j][0]:
                best[j] = (score, i, piece)
for i, b in enumerate(best):
    print(f"best[{i}] = {b}")

pieces, pos = [], n
while pos > 0:
    pieces.append(best[pos][2]); pos = best[pos][1]
print("バックトラック:", pieces[::-1])
print("encode:", sp.encode(text_ja), "->", repr(sp.decode(sp.encode(text_ja))))

best[0] = (0.0, -1, None)
best[1] = (-3.523782253265381, 0, '▁')
best[2] = (-9.469029426574707, 1, '日')
best[3] = (-9.217713356018066, 0, '▁日本')
best[4] = (-13.430808544158936, 1, '日本語')
バックトラック: ['▁', '日本語']
encode: [9, 2481] -> '日本語'


# Embedding

In [14]:
ids = tokenizer.encode("I love you")
print("wte:", tuple(wte.shape), "wpe:", tuple(wpe.shape))

x = wte[ids] + wpe[torch.arange(len(ids))]
print("x:", tuple(x.shape))

# 同じトークンでも、位置が違えば加算後のベクトルは異なる
t = ids[0]
print("位置 0 と位置 2 の wte + wpe が一致:", torch.allclose(wte[t] + wpe[0], wte[t] + wpe[2]))

wte: (50257, 768) wpe: (1024, 768)
x: (3, 768)
位置 0 と位置 2 の wte + wpe が一致: False


# Transformer Block

In [15]:
x = wte[input_ids] + wpe[torch.arange(len(input_ids))]
block = blocks[0]

# TransformerBlock.__call__ と同じ処理
y = x + block.attn(block.ln_1(x))
y = y + block.mlp(block.ln_2(y))
print(torch.allclose(y, block(x)))

True


## LayerNorm

In [16]:
torch.set_printoptions(precision=3, sci_mode=False)

ln = blocks[0].ln_1
print("入力   平均:", x.mean(-1), " 分散:", x.var(-1, correction=0))

eps = 1e-5
mean = x.mean(dim=-1, keepdim=True)
variance = x.var(dim=-1, keepdim=True, correction=0)
x_norm = (x - mean) / torch.sqrt(variance + eps)
print("正規化 平均:", x_norm.mean(-1), " 分散:", x_norm.var(-1, correction=0))

out = ln.g * x_norm + ln.b  # γ でスケール、β でシフト
print("LayerNorm と一致:", torch.allclose(out, ln(x), atol=1e-6))
print("LayerNorm の使用回数:", len(blocks) * 2 + 1)

入力   平均: tensor([-0.007, -0.002, -0.004, -0.002, -0.004])  分散: tensor([0.134, 0.051, 0.036, 0.040, 0.032])
正規化 平均: tensor([-0.000,  0.000, -0.000, -0.000,  0.000])  分散: tensor([1.000, 1.000, 1.000, 1.000, 1.000])
LayerNorm と一致: True
LayerNorm の使用回数: 25


## Attention

### Q, K, V への変換

In [17]:
attn = blocks[0].attn
y = blocks[0].ln_1(x)
seq_len, embed_dim = y.shape
n_head = attn.n_head
d_k = embed_dim // n_head

qkv = y @ attn.w_qkv + attn.b_qkv
print("qkv:", tuple(qkv.shape))
q, k, v = torch.chunk(qkv, 3, dim=-1)
print("q, k, v:", tuple(q.shape))
q = q.reshape(seq_len, n_head, d_k).permute(1, 0, 2)
k = k.reshape(seq_len, n_head, d_k).permute(1, 0, 2)
v = v.reshape(seq_len, n_head, d_k).permute(1, 0, 2)
print("ヘッド分割後:", tuple(q.shape), f"(n_head={n_head}, d_k={d_k})")

qkv: (5, 2304)
q, k, v: (5, 768)
ヘッド分割後: (12, 5, 64) (n_head=12, d_k=64)


### スコア計算

In [18]:
scores = q @ k.transpose(-2, -1) / d_k ** 0.5
print("scores:", tuple(scores.shape))

scores: (12, 5, 5)


### 因果マスキングと Softmax

In [19]:
mask = torch.tril(torch.ones(seq_len, seq_len))
scores = torch.where(mask == 0, torch.tensor(-1e10), scores)
probs = softmax(scores)  # 注目度行列

tokens = [tokenizer.decode([i]) for i in input_ids]
torch.set_printoptions(precision=2)
print("ヘッド 0 の注目度行列（行: 注目する側, 列: 注目される側）")
print(tokens)
print(probs[0])
print("各行の合計:", probs[0].sum(-1))

ヘッド 0 の注目度行列（行: 注目する側, 列: 注目される側）
['The', ' capital', ' of', ' France', ' is']
tensor([[1.00, 0.00, 0.00, 0.00, 0.00],
        [0.85, 0.15, 0.00, 0.00, 0.00],
        [0.70, 0.22, 0.07, 0.00, 0.00],
        [0.61, 0.18, 0.14, 0.07, 0.00],
        [0.61, 0.15, 0.06, 0.09, 0.08]])
各行の合計: tensor([1.00, 1.00, 1.00, 1.00, 1.00])


### Value の重み付き平均と出力射影

In [20]:
out = probs @ v
out = out.permute(1, 0, 2).reshape(seq_len, embed_dim)
out = out @ attn.w_out + attn.b_out
print("Attention と一致:", torch.allclose(out, attn(y), atol=1e-5))

Attention と一致: True


## MLP

In [21]:
mlp = blocks[0].mlp
z = blocks[0].ln_2(x)

i = z @ mlp.w_fc + mlp.b_fc            # 768 → 3072（4倍に拡張）
a = gelu(i)                            # 非線形変換（活性化関数）
out = a @ mlp.w_proj + mlp.b_proj      # 3072 → 768（元に圧縮）
print(tuple(i.shape), "->", tuple(out.shape), "/ MLP と一致:", torch.allclose(out, mlp(z), atol=1e-5))

print("GELU:", gelu(torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])))

(5, 3072) -> (5, 768) / MLP と一致: True
GELU: tensor([-0.00, -0.16,  0.00,  0.84,  3.00])


## 残差接続

In [22]:
# 残差ストリームの観察: 各ブロックが「追加」する量と、ストリーム全体のノルム
x_s = wte[input_ids] + wpe[torch.arange(len(input_ids))]
print(f"embedding: |x|={x_s[-1].norm().item():.1f}")
for i, block in enumerate(blocks):
    a_out = block.attn(block.ln_1(x_s))
    x_s = x_s + a_out
    m_out = block.mlp(block.ln_2(x_s))
    x_s = x_s + m_out
    print(f"block {i:2d}: |attn|={a_out[-1].norm().item():7.1f}  |mlp|={m_out[-1].norm().item():7.1f}  |x|={x_s[-1].norm().item():7.1f}")

embedding: |x|=4.9
block  0: |attn|=   28.4  |mlp|=   32.9  |x|=   56.5
block  1: |attn|=    8.9  |mlp|=    7.3  |x|=   55.2
block  2: |attn|=    8.4  |mlp|=    7.9  |x|=   57.9
block  3: |attn|=   10.1  |mlp|=   13.0  |x|=   64.3
block  4: |attn|=   11.5  |mlp|=   13.6  |x|=   72.0
block  5: |attn|=   11.6  |mlp|=   14.7  |x|=   73.4
block  6: |attn|=   13.0  |mlp|=   23.6  |x|=   85.4
block  7: |attn|=    8.3  |mlp|=   23.9  |x|=   99.6
block  8: |attn|=    8.9  |mlp|=   36.0  |x|=  127.2
block  9: |attn|=   23.8  |mlp|=   45.3  |x|=  165.6
block 10: |attn|=   19.3  |mlp|=  136.0  |x|=  289.9
block 11: |attn|=  200.3  |mlp|=   70.7  |x|=  429.6


## 文脈付き埋め込み

In [23]:
def bank_vector(text):
    ids = tokenizer.encode(text)
    idx = ids.index(tokenizer.encode(" bank")[0])
    h = wte[ids] + wpe[torch.arange(len(ids))]
    emb = wte[ids[idx]]
    for block in blocks:
        h = block(h)
    return emb, h[idx]

def cos(a, b):
    return (a @ b / (a.norm() * b.norm())).item()

emb_r, h_r = bank_vector("I sat on the river bank")
emb_m, h_m = bank_vector("I put money in the bank")
print("Embedding 層 (wte) の一致:", torch.equal(emb_r, emb_m))
print(f"12 ブロック通過後のコサイン類似度: {cos(h_r, h_m):.3f}")

Embedding 層 (wte) の一致: True
12 ブロック通過後のコサイン類似度: 0.937


# 出力とサンプリング

## LM Head（Weight Tying）

In [24]:
x = wte[input_ids] + wpe[torch.arange(len(input_ids))]
for block in blocks:
    x = block(x)
x = ln_f(x)

logits = x @ wte.T  # (seq_len, 768) @ (768, 50257) → (seq_len, 50257)
print("logits:", tuple(logits.shape))
print(f"wte のパラメーター比率: {wte.numel() / 124_439_808:.1%}")

top = torch.topk(logits[-1], 5)
for value, t in zip(top.values, top.indices):
    print(f"{tokenizer.decode([t.item()])!r:12} logit={value.item():.2f}")

logits: (5, 50257)
wte のパラメーター比率: 31.0%
' the'       logit=-100.25
' now'       logit=-100.82
' a'         logit=-100.86
' France'    logit=-101.21
' Paris'     logit=-101.21


## サンプリング手法

In [25]:
last = logits[-1]

def show(label, p, n=5):
    top = torch.topk(p, n)
    print(f"{label:14}", ", ".join(f"{tokenizer.decode([t.item()])!r}:{v.item():.3f}" for v, t in zip(top.values, top.indices)))

# 貪欲法
print("貪欲法:", repr(tokenizer.decode([torch.argmax(last).item()])))

# Temperature
for T in [0.5, 1.0, 2.0]:
    show(f"T={T}", softmax(last / T))

# Top-k
k = 5
top = torch.topk(last, k)
masked = torch.full_like(last, float("-inf"))
masked[top.indices] = top.values
show(f"top-k (k={k})", softmax(masked))

# Top-p
p_thr = 0.9
p = softmax(last)
order = torch.argsort(p, descending=True)
cum = torch.cumsum(p[order], dim=0)
cutoff = torch.searchsorted(cum, p_thr).item() + 1
q = p.clone(); q[order[cutoff:]] = 0.0; q /= q.sum()
print(f"top-p (p={p_thr}) の候補数:", cutoff)
show(f"top-p (p={p_thr})", q)

貪欲法: ' the'
T=0.5          ' the':0.397, ' now':0.128, ' a':0.118, ' France':0.058, ' Paris':0.058
T=1.0          ' the':0.085, ' now':0.048, ' a':0.046, ' France':0.032, ' Paris':0.032
T=2.0          ' the':0.005, ' now':0.004, ' a':0.004, ' France':0.003, ' Paris':0.003
top-k (k=5)    ' the':0.348, ' now':0.197, ' a':0.190, ' France':0.133, ' Paris':0.133
top-p (p=0.9) の候補数: 1503
top-p (p=0.9)  ' the':0.094, ' now':0.053, ' a':0.051, ' France':0.036, ' Paris':0.036


## 自己回帰生成

In [26]:
ids = tokenizer.encode("The capital of France is")
for _ in range(8):
    logits = model(torch.tensor(ids))
    next_token = int(torch.argmax(logits[-1, :]))
    ids.append(next_token)
print(repr(tokenizer.decode(ids)))
naive_ids = list(ids)

'The capital of France is the capital of the French Republic, and'


# KV キャッシュ

In [27]:
ids = tokenizer.encode("The capital of France is")
kv_cache = None  # 初回は None（Prefill）
inputs = torch.tensor(ids)
for step in range(8):
    logits, kv_cache = model(inputs, kv_cache=kv_cache)
    next_token = int(torch.argmax(logits[-1, :]))
    ids.append(next_token)
    inputs = torch.tensor([next_token])  # Incremental: 新トークンのみ処理
    if step == 0:
        print("Prefill 後の K の形状:", tuple(kv_cache[0][0].shape))
print("Incremental 後の K の形状:", tuple(kv_cache[0][0].shape))
print(repr(tokenizer.decode(ids)))
print("キャッシュなしと同じ結果:", ids == naive_ids)

Prefill 後の K の形状: (12, 5, 64)
Incremental 後の K の形状: (12, 12, 64)
'The capital of France is the capital of the French Republic, and'
キャッシュなしと同じ結果: True


In [28]:
# 計算量の比較（プロンプト 4 トークンから 3 トークン生成）
no_cache = sum(n * n for n in [4, 5, 6, 7])
cache = 4 * 4 + sum(1 * n for n in [5, 6, 7])
print("キャッシュなし:", no_cache)
print("キャッシュあり:", cache)

キャッシュなし: 126
キャッシュあり: 34
